# eady1: Eady (1949) baroclinic instability in a zonally periodic channel
pyOM2: for_config/eady0.f90, py_config/eady0.py, py_config/eady1.py, py_config/channel1.py

Linear vertical shear U(z) in thermal wind balance with a meridional buoyancy
gradient, uniform stratification N0, perturbed with the analytical Eady mode of
fastest growth. Buoyancy is the prognostic tracer (eq_of_state=0). With
restoring=True the zonal mean is restored to the initial state so that the
instability can equilibrate.

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PYOM3_ROOT', os.path.abspath('../..')))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.pe_realistic.model import model

In [ ]:
Ri, delta  = 400., 0.01             # Richardson number, aspect ratio
Ek, CFL    = 0.05, 0.02             # Ekman number, CFL number
f0, H0     = 1e-4, 200.
N0         = f0/delta
U0         = N0*H0/np.sqrt(Ri)
M0         = np.sqrt(f0*U0/H0)                           # meridional buoyancy gradient sqrt(M0^2)
om_max     = np.sqrt(5./54)/np.sqrt(1 + Ri)*f0            # growth rate of the fastest growing mode
kmax       = 1./(np.sqrt(2./5)*np.sqrt(1 + Ri)*M0**2/f0**2*H0)   # its wavenumber
nx, ny, nz = 60, 60, 20
Lx = Ly    = 2*2*np.pi/kmax                               # two wavelengths
dx, dz     = Lx/nx, H0/nz
dt         = CFL/U0*dx
b_amp      = 1e-5                   # amplitude of the perturbation
restoring  = True
t_rest     = om_max/5
nsteps     = int(45*86400/dt)       # 45 days
plot_every = 200
print('dt = %.0f s, growth time 1/om_max = %.1f days, %d steps' % (dt, 1/om_max/86400, nsteps))

par = dict(nx=nx, ny=ny, nz=nz, dxt=dx, dyt=dx, dzt=dz, dt=dt,
           coord_degree=False, set_zero_at_surface=True,
           periodic_in_x=True, periodic_in_y=False,
           eq_of_state=0, implicit_free_surface=False, explicit_free_surface=False,
           epsilon=1e-12*(dx/20e3)**2, max_iterations=5000, superbee_advection=True,
           Ahbi=Ek*f0*dx**4, vmix_scheme='explicit', A_v=Ek*f0*dz**2)

In [ ]:
class eady1(model):

    def set_coriolis(self):
        self.coriolis_t = OM.np.asarray(f0 + 0*self.yt[:, None] + 0*self.xt[None, :])

    def set_initial_conditions(self):
        Z, Y, X = np.meshgrid(self.zt_gl, self.yt_gl, self.xt_gl, indexing='ij')
        u0 = U0/2 + U0*Z/H0
        b0 = N0**2*Z - f0*U0/H0*Y
        # Eady mode of fastest growth
        d    = f0/N0/np.sqrt(kmax**2 + (np.pi/Ly)**2)
        c1   = (1 + 0.25*(H0/d)**2 - H0/d/np.tanh(H0/d))*complex(1, 0)
        c1   = (np.sqrt(c1)*d/H0 + 0.5)*U0
        A    = (U0 - c1)/U0*H0/d
        phiz = abs(A/d*np.sinh(Z/d) + np.cosh(Z/d)/d)
        b    = b0 + b_amp*phiz/phiz.max()*np.sin(kmax*X)*np.sin(np.pi*Y/Ly)
        zeros = np.zeros((nz, ny, nx))
        super().set_initial_conditions(u0, zeros, b, zeros, np.zeros((ny, nx)))
        Z, Y, X = np.meshgrid(self.zt, self.yt, self.xt, indexing='ij')
        self.u0 = OM.np.asarray(U0/2 + U0*Z/H0)*self.maskU
        self.b0 = OM.np.asarray(N0**2*Z - f0*U0/H0*Y)*self.maskT

    def set_forcing(self, tau):
        if not restoring: return
        h  = self.halo                         # zonal mean over the interior
        um = OM.np.mean(self.u[:, :, h:-h], axis=2)[:, :, None]
        bm = OM.np.mean(self.temp[:, :, h:-h], axis=2)[:, :, None]
        self.du    = OM.add_to_array(self.du, (tau,), t_rest*(self.u0 - um)*self.maskU)
        self.dtemp = OM.add_to_array(self.dtemp, (tau,), t_rest*(self.b0 - bm)*self.maskT)

In [ ]:
M = eady1(par)
M.set_coriolis()
M.set_topography()
M.prepare_for_run()
M.set_initial_conditions()
h = M.halo
ks = -h - 1                         # surface level
rec = []                            # time and rms of v

In [ ]:
def plot(t, tau):
    clear_output(wait=True)
    rec.append((t, np.sqrt(np.mean(M.v[h:-h, h:-h, h:-h]**2))))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    c = ax[0].pcolormesh(M.xt_gl/1e3, M.yt_gl/1e3, M.temp[ks, h:-h, h:-h], cmap='RdBu_r')
    plt.colorbar(c, ax=ax[0], label='surface b [m/s$^2$]')
    c = ax[1].pcolormesh(M.xt_gl/1e3, M.yt_gl/1e3, M.v[ks, h:-h, h:-h], cmap='RdBu_r')
    plt.colorbar(c, ax=ax[1], label='surface v [m/s]')
    for a in ax: a.set_xlabel('x [km]'); a.set_ylabel('y [km]')
    fig.suptitle('t = %.1f days' % (t/86400)); plt.tight_layout(); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step_implicit_pressure(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt, tau)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
# growth of the perturbation compared with the Eady growth rate
t = np.array([a[0] for a in rec]); v = np.array([a[1] for a in rec])
plt.figure(figsize=(6, 4))
plt.semilogy(t/86400, v, label='rms v')
plt.semilogy(t/86400, v[1]*np.exp(om_max*(t - t[1])), 'k--', label='exp(om_max t)')
plt.xlabel('t [days]'); plt.ylabel('[m/s]'); plt.legend(); plt.show()